# VAE Training Data Setup

Purpose: Generate and compare the curated, random, and mixed attack-profile datasets used by the VAE training workflow.

This notebook is intentionally dataset-focused. It does not train the VAE. It produces train/valid/test JSONL files, summarizes the three source types, and gives quick visual checks before moving to `notebooks/vae_train.ipynb`.


## Imports And Repo Setup

In [ ]:
from __future__ import annotations

from collections import Counter
from copy import deepcopy
import json
from pathlib import Path
import sys
import time
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "convoy_sim").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments.generate_attack_profile_scaffold import (
    HIT_THREAT_LABEL,
    INTENTIONAL_MISS_LABEL,
    MIN_SPAWN_CLEARANCE_M,
    NEAR_MISS_LABEL,
    generate_attack_profile_scaffolds,
    render_profiles_as_jsonl,
)
from experiments.generate_random_attack_profile_dataset import generate_random_baseline_records
from experiments.build_mixed_attack_profile_dataset import build_mixed_split
from convoy_sim.profile_generation_viz import (
    FIXED_X_LIMITS,
    apply_limits,
    combined_limits,
    simple_ship_polygons,
    style_ax,
)
from scenarios.convoy_profiles import get_convoy_layout_profile


## Configuration

Default counts create a 55k-record source set: 45k train, 5k validation, and 5k test. Set `OVERWRITE_EXISTING = False` if you want the notebook to stop before replacing existing JSONLs.


In [ ]:
NOTEBOOK_NAME = "profile_generation_tests"
CONVOY_PROFILE = "convoy_layout_1"
DATA_DIR = PROJECT_ROOT / "data" / "attack_profiles" / "synthetic"

TRAIN_COUNT = 45_000
VALID_COUNT = 5_000
TEST_COUNT = 5_000
SPLITS = {
    "train": {"count": TRAIN_COUNT, "seed_offset": 0, "start_index": 1},
    "valid": {"count": VALID_COUNT, "seed_offset": 10, "start_index": 1},
    "test": {"count": TEST_COUNT, "seed_offset": 20, "start_index": 1},
}

BASE_SEED = 1945
CURATED_FRACTION = 0.70

RUN_CURATED_GENERATION = True
RUN_RANDOM_GENERATION = True
RUN_MIXED_BUILD = True
WRITE_DATASETS = True
OVERWRITE_EXISTING = True

PLOT_SAMPLE_PER_SOURCE = 5_000
PLOT_RANDOM_SEED = 1945

LABEL_ORDER = [HIT_THREAT_LABEL, NEAR_MISS_LABEL, INTENTIONAL_MISS_LABEL]
EXPECTED_LABEL_FRACTIONS = {
    HIT_THREAT_LABEL: 0.65,
    NEAR_MISS_LABEL: 0.25,
    INTENTIONAL_MISS_LABEL: 0.10,
}
MIN_EXPECTED_SPAWN_REGIONS = 3
MIN_EXPECTED_APPROACH_SIDES = 3
MIN_CLEARANCE_M = MIN_SPAWN_CLEARANCE_M

PATHS = {
    "curated_v4": {
        "train": DATA_DIR / "train_random_tactical_v4_45k.jsonl",
        "valid": DATA_DIR / "valid_random_tactical_v4_5k.jsonl",
        "test": DATA_DIR / "test_random_tactical_v4_5k.jsonl",
    },
    "random_v1": {
        "train": DATA_DIR / "train_random_profile_v1_45k.jsonl",
        "valid": DATA_DIR / "valid_random_profile_v1_5k.jsonl",
        "test": DATA_DIR / "test_random_profile_v1_5k.jsonl",
    },
    "mixed_70_30": {
        "train": DATA_DIR / "train_mixed_curated70_random30_45k.jsonl",
        "valid": DATA_DIR / "valid_mixed_curated70_random30_5k.jsonl",
        "test": DATA_DIR / "test_mixed_curated70_random30_5k.jsonl",
    },
}


## Helpers

In [ ]:
def write_jsonl(path: Path, records: list[dict[str, Any]], *, overwrite: bool) -> None:
    if path.exists() and not overwrite:
        raise FileExistsError(f"{path} exists. Set OVERWRITE_EXISTING = True to replace it.")
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, sort_keys=True) + "\n")


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def profile_id(record: dict[str, Any]) -> str:
    return str(record.get("profile", {}).get("profile_id", ""))


def record_label(record: dict[str, Any]) -> str:
    return str(
        record.get("intent", {}).get("intended_label")
        or record.get("audit", {}).get("suggested_label")
        or record.get("outcome", {}).get("actual_outcome_label")
        or "unknown"
    )


def record_spawn_region(record: dict[str, Any]) -> str:
    return str(
        record.get("intent", {}).get("spawn_region")
        or record.get("audit", {}).get("spawn_region")
        or record.get("outcome", {}).get("spawn_region")
        or "unknown"
    )


def record_approach_side(record: dict[str, Any]) -> str:
    return str(
        record.get("intent", {}).get("approach_side")
        or record.get("audit", {}).get("approach_side")
        or record.get("outcome", {}).get("approach_side")
        or "unknown"
    )


def record_target_zone_kind(record: dict[str, Any]) -> str:
    return str(
        record.get("intent", {}).get("target_zone_kind")
        or record.get("audit", {}).get("target_zone_kind")
        or record.get("outcome", {}).get("target_zone_kind")
        or "unknown"
    )


def record_clearance(record: dict[str, Any]) -> float:
    value = (
        record.get("intent", {}).get("nearest_ship_clearance_m")
        or record.get("audit", {}).get("clearance_m")
        or record.get("outcome", {}).get("clearance_m")
    )
    return float(value) if value is not None else float("nan")


def record_xy(record: dict[str, Any]) -> tuple[float, float]:
    u_pos = record["profile"]["u_pos"]
    return float(u_pos[0]), float(u_pos[1])


def write_summary(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def sample_records(records: list[dict[str, Any]], *, n: int, seed: int) -> list[dict[str, Any]]:
    if len(records) <= int(n):
        return list(records)
    rng = np.random.default_rng(int(seed))
    indices = rng.choice(np.arange(len(records)), size=int(n), replace=False)
    return [records[int(idx)] for idx in indices]


## Generate Curated V4 Splits

Curated v4 is the tactical, spawn-first generator and remains the realism anchor for VAE work.


In [ ]:
def generate_curated_v4_records(*, count: int, seed: int, start_index: int, split: str) -> list[dict[str, Any]]:
    profiles, audit_rows = generate_attack_profile_scaffolds(
        mode="random_tactical_v4",
        convoy_profile=CONVOY_PROFILE,
        count=int(count),
        seed=int(seed),
        start_index=int(start_index),
    )
    rendered = render_profiles_as_jsonl(
        profiles,
        audit_rows=audit_rows,
        seed=int(seed),
        convoy_profile=CONVOY_PROFILE,
        accepted_labels=LABEL_ORDER,
        mode="random_tactical_v4",
    )
    records = [json.loads(line) for line in rendered.splitlines() if line.strip()]
    for idx, record in enumerate(records):
        record.setdefault("generator_meta", {})["split"] = split
        record.setdefault("generator_meta", {})["dataset_role"] = "vae_training_data"
        record.setdefault("generator_meta", {})["source_family"] = "curated_v4"
    return records


curated_splits: dict[str, list[dict[str, Any]]] = {}
curated_timings = []
if RUN_CURATED_GENERATION:
    for split, cfg in SPLITS.items():
        start = time.perf_counter()
        records = generate_curated_v4_records(
            count=int(cfg["count"]),
            seed=int(BASE_SEED + cfg["seed_offset"]),
            start_index=int(cfg["start_index"]),
            split=split,
        )
        curated_splits[split] = records
        seconds = time.perf_counter() - start
        curated_timings.append({"source": "curated_v4", "split": split, "records": len(records), "seconds": seconds})
        if WRITE_DATASETS:
            write_jsonl(PATHS["curated_v4"][split], records, overwrite=OVERWRITE_EXISTING)
        print(f"curated_v4 {split}: {len(records):,} records in {seconds:.1f}s")
else:
    curated_splits = {split: read_jsonl(path) for split, path in PATHS["curated_v4"].items()}

pd.DataFrame(curated_timings)


## Generate Random V1 Splits

Random v1 is the profile-first baseline. It is broader and less curated, but it uses the same no-collision and dynamic outcome gate standards.


In [ ]:
random_splits: dict[str, list[dict[str, Any]]] = {}
random_timings = []
random_stats_by_split = {}
if RUN_RANDOM_GENERATION:
    for split, cfg in SPLITS.items():
        start = time.perf_counter()
        records, stats = generate_random_baseline_records(
            count=int(cfg["count"]),
            seed=int(BASE_SEED + 100 + cfg["seed_offset"]),
            start_index=int(cfg["start_index"]),
            convoy_profile=CONVOY_PROFILE,
        )
        for record in records:
            record.setdefault("generator_meta", {})["split"] = split
            record.setdefault("generator_meta", {})["dataset_role"] = "vae_training_data"
            record.setdefault("generator_meta", {})["source_family"] = "random_v1"
        random_splits[split] = records
        random_stats_by_split[split] = stats
        seconds = time.perf_counter() - start
        random_timings.append({
            "source": "random_v1",
            "split": split,
            "records": len(records),
            "seconds": seconds,
            "acceptance_rate": float(stats.get("acceptance_rate", 0.0)),
            "attempts": int(stats.get("attempts", 0)),
        })
        if WRITE_DATASETS:
            write_jsonl(PATHS["random_v1"][split], records, overwrite=OVERWRITE_EXISTING)
        print(f"random_v1 {split}: {len(records):,} records in {seconds:.1f}s")
else:
    random_splits = {split: read_jsonl(path) for split, path in PATHS["random_v1"].items()}

pd.DataFrame(random_timings)


## Build Mixed 70/30 Splits

The mixed source samples each split independently from the matching curated and random split. This avoids train/valid/test leakage.


In [ ]:
mixed_splits: dict[str, list[dict[str, Any]]] = {}
mixed_summaries = {}
if RUN_MIXED_BUILD:
    for split, cfg in SPLITS.items():
        records, summary = build_mixed_split(
            curated_records=curated_splits[split],
            random_records=random_splits[split],
            total_count=int(cfg["count"]),
            curated_fraction=float(CURATED_FRACTION),
            split=split,
            seed=int(BASE_SEED + 200 + cfg["seed_offset"]),
        )
        mixed_splits[split] = records
        mixed_summaries[split] = summary
        if WRITE_DATASETS:
            write_jsonl(PATHS["mixed_70_30"][split], records, overwrite=OVERWRITE_EXISTING)
            write_summary(PATHS["mixed_70_30"][split].with_suffix(PATHS["mixed_70_30"][split].suffix + ".summary.json"), summary)
        print(f"mixed_70_30 {split}: {len(records):,} records | {summary['source_counts']}")
else:
    mixed_splits = {split: read_jsonl(path) for split, path in PATHS["mixed_70_30"].items()}

datasets = {
    "curated_v4": curated_splits,
    "random_v1": random_splits,
    "mixed_70_30": mixed_splits,
}

pd.DataFrame([
    {
        "split": split,
        "total_count": summary["total_count"],
        "curated_count": summary["source_counts"].get("curated_v4", 0),
        "random_count": summary["source_counts"].get("random_profile_v1", 0),
        "curated_fraction": summary["source_counts"].get("curated_v4", 0) / max(summary["total_count"], 1),
    }
    for split, summary in mixed_summaries.items()
])


## Dataset Summary Tables

In [ ]:
def summarize_records(source: str, split: str, records: list[dict[str, Any]]) -> dict[str, Any]:
    labels = Counter(record_label(record) for record in records)
    spawn_regions = Counter(record_spawn_region(record) for record in records)
    approach_sides = Counter(record_approach_side(record) for record in records)
    target_zone_kinds = Counter(record_target_zone_kind(record) for record in records)
    clearances = np.asarray([record_clearance(record) for record in records], dtype=float)
    ids = [profile_id(record) for record in records]
    mixed_sources = Counter(str(record.get("mixture_meta", {}).get("source_dataset", "")) for record in records)
    mixed_sources.pop("", None)
    return {
        "source": source,
        "split": split,
        "records": int(len(records)),
        "unique_profile_ids": int(len(set(ids))),
        "duplicate_profile_ids": int(len(ids) - len(set(ids))),
        "hit_count": int(labels.get(HIT_THREAT_LABEL, 0)),
        "near_miss_count": int(labels.get(NEAR_MISS_LABEL, 0)),
        "intentional_miss_count": int(labels.get(INTENTIONAL_MISS_LABEL, 0)),
        "hit_fraction": float(labels.get(HIT_THREAT_LABEL, 0) / max(len(records), 1)),
        "near_miss_fraction": float(labels.get(NEAR_MISS_LABEL, 0) / max(len(records), 1)),
        "intentional_miss_fraction": float(labels.get(INTENTIONAL_MISS_LABEL, 0) / max(len(records), 1)),
        "spawn_regions": int(len(spawn_regions)),
        "approach_sides": int(len(approach_sides)),
        "target_zone_kinds": int(len(target_zone_kinds)),
        "inside_fraction": float(spawn_regions.get("inside_convoy_envelope", 0) / max(len(records), 1)),
        "clearance_min_m": float(np.nanmin(clearances)) if len(clearances) else float("nan"),
        "clearance_mean_m": float(np.nanmean(clearances)) if len(clearances) else float("nan"),
        "mixed_curated_count": int(mixed_sources.get("curated_v4", 0)),
        "mixed_random_count": int(mixed_sources.get("random_profile_v1", 0)),
    }

summary_rows = [
    summarize_records(source, split, records)
    for source, split_map in datasets.items()
    for split, records in split_map.items()
]
summary_df = pd.DataFrame(summary_rows).sort_values(["source", "split"]).reset_index(drop=True)
summary_df


In [ ]:
label_rows = []
for source, split_map in datasets.items():
    for split, records in split_map.items():
        counts = Counter(record_label(record) for record in records)
        for label in LABEL_ORDER:
            label_rows.append({
                "source": source,
                "split": split,
                "label": label,
                "count": int(counts.get(label, 0)),
                "fraction": float(counts.get(label, 0) / max(len(records), 1)),
            })
label_df = pd.DataFrame(label_rows)
label_df.pivot_table(index=["source", "split"], columns="label", values="fraction", fill_value=0.0)


In [ ]:
spawn_rows = []
for source, split_map in datasets.items():
    for split, records in split_map.items():
        counts = Counter(record_spawn_region(record) for record in records)
        for region, count in sorted(counts.items()):
            spawn_rows.append({"source": source, "split": split, "spawn_region": region, "count": int(count)})
spawn_df = pd.DataFrame(spawn_rows)
spawn_df.loc[spawn_df["split"] == "train"].pivot_table(index="spawn_region", columns="source", values="count", fill_value=0).astype(int)


## VAE Readiness Checks

These checks are intentionally simple. They catch broken splits and obvious coverage/collision failures without re-running every historical generator diagnostic.


In [ ]:
def readiness_row(row: pd.Series) -> dict[str, Any]:
    source = str(row["source"])
    split = str(row["split"])
    expected_count = int(SPLITS[split]["count"])
    count_ok = int(row["records"]) == expected_count
    ids_ok = int(row["duplicate_profile_ids"]) == 0
    clearance_ok = float(row["clearance_min_m"]) >= float(MIN_CLEARANCE_M)
    region_ok = int(row["spawn_regions"]) >= MIN_EXPECTED_SPAWN_REGIONS
    approach_ok = int(row["approach_sides"]) >= MIN_EXPECTED_APPROACH_SIDES
    label_total_ok = int(row["hit_count"] + row["near_miss_count"] + row["intentional_miss_count"]) == int(row["records"])
    mixed_ok = True
    if source == "mixed_70_30":
        mixed_ok = abs(float(row["mixed_curated_count"]) / max(int(row["records"]), 1) - float(CURATED_FRACTION)) <= 0.02
    ready = bool(count_ok and ids_ok and clearance_ok and region_ok and approach_ok and label_total_ok and mixed_ok)
    return {
        "source": source,
        "split": split,
        "records": int(row["records"]),
        "count_ok": bool(count_ok),
        "ids_ok": bool(ids_ok),
        "clearance_ok": bool(clearance_ok),
        "region_ok": bool(region_ok),
        "approach_ok": bool(approach_ok),
        "labels_ok": bool(label_total_ok),
        "mixed_ratio_ok": bool(mixed_ok),
        "ready": bool(ready),
    }

readiness_df = pd.DataFrame([readiness_row(row) for _, row in summary_df.iterrows()])
readiness_df


In [ ]:
if not bool(readiness_df["ready"].all()):
    display(readiness_df.loc[~readiness_df["ready"]])
    raise AssertionError("At least one source/split failed VAE readiness checks.")
print("All generated source/split datasets passed the lightweight VAE readiness checks.")


## Combined Train Spawn Plots

These plots sample the train splits for readability. Tables above use the full datasets.


In [ ]:
ships = get_convoy_layout_profile(CONVOY_PROFILE).build_ships()
train_samples = {
    source: sample_records(split_map["train"], n=int(PLOT_SAMPLE_PER_SOURCE), seed=int(PLOT_RANDOM_SEED))
    for source, split_map in datasets.items()
}
all_sample_xy = np.vstack([
    np.asarray([record_xy(record) for record in records], dtype=float)
    for records in train_samples.values()
    if records
])
shared_limits = combined_limits(ships, extra_points=all_sample_xy, pad=350.0)

region_order = sorted({record_spawn_region(record) for records in train_samples.values() for record in records})
colors = plt.cm.tab10(np.linspace(0, 1, max(len(region_order), 1)))
region_color_map = dict(zip(region_order, colors))

fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor="lightgrey", sharex=True, sharey=True)
for ax, (source, records) in zip(axes, train_samples.items()):
    style_ax(ax, f"{source} train spawns")
    simple_ship_polygons(ax, ships)
    for region in region_order:
        points = np.asarray([record_xy(record) for record in records if record_spawn_region(record) == region], dtype=float)
        if len(points):
            ax.scatter(points[:, 0], points[:, 1], s=8, alpha=0.45, color=region_color_map[region], label=region, zorder=2)
    apply_limits(ax, shared_limits, fixed_x_limits=FIXED_X_LIMITS)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
handles, labels = axes[-1].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="lower center", ncol=min(len(labels), 5), frameon=False)
fig.subplots_adjust(bottom=0.22)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor="lightgrey", sharex=True, sharey=True)
for ax, (source, records) in zip(axes, train_samples.items()):
    style_ax(ax, f"{source} train spawns, single color")
    simple_ship_polygons(ax, ships)
    points = np.asarray([record_xy(record) for record in records], dtype=float)
    if len(points):
        ax.scatter(points[:, 0], points[:, 1], s=8, alpha=0.35, color="#111111", label="U-boat spawn", zorder=2)
    apply_limits(ax, shared_limits, fixed_x_limits=FIXED_X_LIMITS)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
handles, labels = axes[-1].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="lower center", ncol=1, frameon=False)
fig.subplots_adjust(bottom=0.18)
plt.show()


## Individual Train Spawn Plots

In [ ]:
for source, records in train_samples.items():
    fig, ax = plt.subplots(figsize=(9, 7), facecolor="lightgrey")
    style_ax(ax, f"{source} train spawns by region")
    simple_ship_polygons(ax, ships)
    for region in region_order:
        points = np.asarray([record_xy(record) for record in records if record_spawn_region(record) == region], dtype=float)
        if len(points):
            ax.scatter(points[:, 0], points[:, 1], s=9, alpha=0.5, color=region_color_map[region], label=region, zorder=2)
    apply_limits(ax, shared_limits, fixed_x_limits=FIXED_X_LIMITS)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
    plt.tight_layout()
    plt.show()


## Output Paths

In [ ]:
path_rows = []
for source, split_map in PATHS.items():
    for split, path in split_map.items():
        path_rows.append({
            "source": source,
            "split": split,
            "path": str(path.relative_to(PROJECT_ROOT)),
            "exists": path.exists(),
            "size_mb": path.stat().st_size / 1_000_000 if path.exists() else 0.0,
        })
paths_df = pd.DataFrame(path_rows).sort_values(["source", "split"]).reset_index(drop=True)
paths_df


## Next Step

If the readiness table passes and the train-spawn plots look reasonable, move to `notebooks/vae_train.ipynb` and train the curated, random, and mixed VAE configurations against these files.
